# MF-HGNN ABIDE Reproduction Guide

This notebook includes:

1. Data loading and a brief overview
2. Dataset splitting (label-stratified K-fold) and summary
3. Full training procedure (calling `main.py`)
4. Evaluation with the best saved models and viewing logs

All core training and evaluation logic is identical to the original experimental code used in the paper, implemented via `main.py`, `dataload.py`, `model.py`, etc.


In [1]:
# Environment and dependencies

import os
import numpy as np
import torch

from opt import OptInit
from dataload import dataloader

# Initialize hyperparameters (same as running main.py from the command line)
opt = OptInit().initialize()
print('Device:', opt.device)

# Print several important paths for this notebook
print('subject_IDs_path:', opt.subject_IDs_path)
print('phenotype_path:', opt.phenotype_path)
print('data_path:', opt.data_path)
print('log_path:', opt.log_path)
print('ckpt_path:', opt.ckpt_path)


 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is train.
 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is

## 1. Data loading and overview

In this section, we use the `dataloader` class in `dataload.py` to reproduce exactly the same data-loading pipeline as in the official experiments, and briefly inspect key data structures:

- Raw node features `raw_features`
- Label vector `y`
- Phenotypic data `phonetic_data` (site, sex, age, etc.)
- Dynamic functional connectivity `dynamic_fc`


In [2]:
# 1. Data loading (detailed prints)

dl = dataloader()
raw_features, y, nonimg, phonetic_score, time_series, dynamic_fc = dl.load_data()

# Avoid flooding the notebook output; only show parts of large arrays
np.set_printoptions(threshold=20)

# Basic information
print('Number of raw_features samples:', len(raw_features))
print('Shape of labels y:', np.array(y).shape)
print('Shape of non-image phenotypes nonimg:', np.array(nonimg).shape)
print('Number of dynamic_fc samples:', len(dynamic_fc))

# Inspect the first sample
first_feat = raw_features[0]
print('\nType of the first sample:', type(first_feat))
if hasattr(first_feat, 'shape'):
    print('Shape of the first sample features:', first_feat.shape)
elif hasattr(first_feat, 'x'):
    # For torch_geometric.data.Data
    print('Shape of node features x of the first sample:', first_feat.x.shape)
    print('Shape of edge_index of the first sample:', first_feat.edge_index.shape)

# Label distribution
unique, counts = np.unique(y, return_counts=True)
print('\nLabel distribution (label: count):')
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

# Phenotypic information (SITE_ID, SEX, AGE_AT_SCAN)
print('\nSITE_ID for all samples:')
print(phonetic_score['SITE_ID'])

print('\nSEX for all samples:')
print(phonetic_score['SEX'])

print('\nAGE_AT_SCAN for all samples:')
print(phonetic_score['AGE_AT_SCAN'])


Number of raw_features samples: 871
Shape of labels y: (871,)
Shape of non-image phenotypes nonimg: (871, 3)
Number of dynamic_fc samples: 871

Type of the first sample: <class 'torch_geometric.data.data.Data'>
Shape of node features x of the first sample: torch.Size([111, 111])
Shape of edge_index of the first sample: torch.Size([2, 5358])

Label distribution (label: count):
  0.0: 403
  1.0: 468

SITE_ID for all samples:
[ 0.  0.  0. ... 19. 19. 19.]

SEX for all samples:
[1. 2. 1. ... 1. 2. 1.]

AGE_AT_SCAN for all samples:
[37.7  20.2  20.9  ... 11.08  9.5  14.42]


## 2. Dataset splitting strategy in the current code

In this repository, dataset splitting is implemented in `dataload.py` via `dataloader.data_split()`. The key idea is:

- Perform label-stratified K-fold cross-validation at the **subject level**.
- Use `StratifiedKFold`, stratifying on the diagnosis label `y` (DX_GROUP).
- In other words, when creating training/test folds, the class proportions (label distribution) in each fold are kept similar to the overall dataset, but there is **no explicit stratification on site `SITE_ID`**.
- As a result, the numbers of positive/negative samples in each fold are close to the global distribution, while the site distribution across folds may vary; this is the actual splitting strategy used by the current code.

The following code cell calls `dataloader.data_split` from this directory and prints, for each fold, the number of samples and the distribution of `(label, SITE_ID)` combinations, providing a clear view of how different sites and diagnosis labels are distributed across folds under this label-only stratification strategy.


In [3]:
# 2*. Detailed illustration of the current splitting strategy

n_folds = opt.n_folds
cv_splits = dl.data_split(n_folds)

labels = np.array(y)
sites = np.array(phonetic_score['SITE_ID'])

print(f'Total number of samples: {len(labels)}')
print(f'Number of folds n_folds: {n_folds}\n')

for fold, (train_idx, test_idx) in enumerate(cv_splits):
    y_tr, s_tr = labels[train_idx], sites[train_idx]
    y_te, s_te = labels[test_idx], sites[test_idx]

    print(f'====== Fold {fold} ======')
    print(f'Number of training samples: {len(train_idx)}, number of test samples: {len(test_idx)}')

    # Distribution of (label, SITE_ID) combinations in training and test sets
    comb_tr = y_tr * 1000 + s_tr
    comb_te = y_te * 1000 + s_te
    u_tr, c_tr = np.unique(comb_tr, return_counts=True)
    u_te, c_te = np.unique(comb_te, return_counts=True)

    print('Training set (label*1000 + SITE_ID) combination distribution:')
    print(dict(zip(u_tr.astype(int), c_tr)))
    print('Test set (label*1000 + SITE_ID) combination distribution:')
    print(dict(zip(u_te.astype(int), c_te)))
    print()


Total number of samples: 871
Number of folds n_folds: 10

====== Fold 0 ======
Number of training samples: 783, number of test samples: 88
Training set (label*1000 + SITE_ID) combination distribution:
{0: 4, 1: 6, 2: 10, 3: 12, 4: 10, 5: 17, 6: 65, 7: 11, 8: 13, 9: 24, 10: 12, 11: 7, 12: 11, 13: 15, 14: 33, 15: 10, 16: 28, 17: 13, 18: 39, 19: 22, 1000: 10, 1001: 5, 1002: 19, 1003: 12, 1004: 14, 1005: 25, 1006: 91, 1007: 11, 1008: 12, 1009: 24, 1010: 13, 1011: 18, 1012: 12, 1013: 22, 1014: 25, 1015: 7, 1016: 46, 1017: 17, 1018: 22, 1019: 16}
Test set (label*1000 + SITE_ID) combination distribution:
{0: 1, 2: 2, 3: 2, 4: 2, 5: 2, 6: 9, 7: 1, 8: 1, 11: 1, 12: 1, 13: 4, 14: 4, 15: 1, 16: 6, 18: 4, 1002: 2, 1003: 2, 1004: 2, 1005: 2, 1006: 7, 1007: 2, 1008: 2, 1009: 2, 1010: 1, 1011: 1, 1012: 1, 1013: 3, 1014: 2, 1015: 3, 1016: 6, 1017: 4, 1018: 2, 1019: 3}

====== Fold 1 ======
Number of training samples: 784, number of test samples: 87
Training set (label*1000 + SITE_ID) combination distr

## 3. Training example (calling main.py)

The official training and evaluation pipeline is implemented in `main.py` under `if __name__ == "__main__":`, which includes:

- Initializing hyperparameters (`OptInit`)
- Calling `dataloader` to load the data
- Running label-stratified K-fold cross-validation
- Printing training and test metrics at each epoch
- Saving the best model (on the test set) for each fold to `opt.ckpt_path`

In this notebook we invoke `main.py` via a command-line call so that the behavior is exactly the same as running the script directly, and a **training log file** is automatically generated (handled by the `Logger` class).

> Training from scratch can be time-consuming, so in practice we usually pre-train on the server, save the best models and logs, and then use this notebook mainly for demonstration.


In [ ]:
import sys

# Use the Python interpreter of the current notebook kernel
!{sys.executable} main.py --train 1 --num_iter 400 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./run.log


In [4]:
# 3.2 View a snippet of the training log

log_path = './run.log'

if os.path.exists(log_path):
    print(f'Reading log file: {log_path}\n')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show only the last 40 lines for readability
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Log file does not exist. Please run the training cell above first.')


Reading log file: ./run.log

Epoch: 470,	ce loss: 0.02003,	ce loss_cla: 0.02003,	train acc: 0.99235,	test acc: 0.90805,	test spe: 0.95000,	test sen: 0.87234      2025-05-10 01:53:42
Epoch: 471,	ce loss: 0.01901,	ce loss_cla: 0.01901,	train acc: 0.99235,	test acc: 0.90805,	test spe: 0.95000,	test sen: 0.87234      2025-05-10 01:54:51
Epoch: 472,	ce loss: 0.02104,	ce loss_cla: 0.02104,	train acc: 0.99235,	test acc: 0.90805,	test spe: 0.95000,	test sen: 0.87234      2025-05-10 01:55:58
Epoch: 473,	ce loss: 0.01533,	ce loss_cla: 0.01533,	train acc: 0.99490,	test acc: 0.90805,	test spe: 0.95000,	test sen: 0.87234      2025-05-10 01:57:07
Epoch: 474,	ce loss: 0.01913,	ce loss_cla: 0.01913,	train acc: 0.99235,	test acc: 0.89655,	test spe: 0.95000,	test sen: 0.85106      2025-05-10 01:58:13
Epoch: 475,	ce loss: 0.01272,	ce loss_cla: 0.01272,	train acc: 0.99362,	test acc: 0.88506,	test spe: 0.97500,	test sen: 0.80851      2025-05-10 01:59:21
Epoch: 476,	ce loss: 0.01146,	ce loss_cla: 0.01146,	t

## 4. Evaluation with the best saved models

In `main.py`, the `evaluate()` function is responsible for:

- Loading the best-performing model for each fold from `opt.ckpt_path`.
- Running forward passes on the corresponding test set.
- Computing and printing multiple metrics (accuracy, AUC, sensitivity, specificity, F1-score, etc.).

Here we call the same script with `--train` set to `0` to perform evaluation using the saved models.

> Before evaluation, make sure that the corresponding model checkpoints for all folds have been successfully saved in `ckpt_path` during training.


In [5]:
import sys

# 4.1 Example: evaluate using pre-trained models

# Assume the training stage used ./ckpt_demo/ as ckpt_path.
# Here we keep the same ckpt_path, set train to 0, and run evaluation only.

!{sys.executable} main.py --train 0 --num_iter 500 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./test.log


 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is eval.
 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> P

In [6]:
# 4.2 View a snippet of the evaluation log

log_path_test = './test.log'

if os.path.exists(log_path_test):
    print(f'Reading evaluation log file: {log_path_test}\n')
    with open(log_path_test, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show the last 40 lines of evaluation results
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Evaluation log file does not exist. Please run the evaluation cell above first.')


Reading evaluation log file: ./test.log

    (convs4): ModuleList(
      (0): TransformerConv(2000, 20, heads=1)
      (1-4): 4 x TransformerConv(40, 20, heads=1)
    )
    (bns): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns2): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns3): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (out_fc): Linear(in_features=100, out_features=2, bias=True)
    (conv5): Linear(in_features=20, out_features=1, bias=False)
    (conv6): Linear(in_features=20, out_features=1, bias=False)
    (conv7): Linear(in_features=20, out_features=1, bias=False)
    (conv8): Linear(in_features=20, out_features=1, bias=False)
    (conv9): Linear(in_features=20, out_features=1, bias=False)
    (conv10): Linear(in_features=20, out_features=1, bias=False)
 